In [12]:
import os, sys
import numpy as np
import cv2
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [13]:
DATA_ROOT = r'C:\hong\python-ws\project-1\dataset\data 2'
RESULT_PATH = r'C:\hong\python-ws\project-1\result.csv'
IMG_HEIGHT, IMG_WIDTH, CHANNELS = 30, 30, 3
NUM_CLASSES = 43
BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
SEED = 42
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f'Device: {DEVICE}')

Device: cuda


In [14]:
print('\n[1] Loading training data...')
data, labels = [], []
train_path = os.path.join(DATA_ROOT, 'Train')

for i in range(NUM_CLASSES):
    class_dir = os.path.join(train_path, str(i))
    if not os.path.isdir(class_dir):
        continue
    files = [f for f in os.listdir(class_dir) if f.endswith('.png')]
    for fname in files:
        img = cv2.imread(os.path.join(class_dir, fname))
        if img is None:
            continue
        img = Image.fromarray(img, 'RGB')
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))
        data.append(np.array(img))
        labels.append(i)

data = np.array(data, dtype=np.float32) / 255.0
labels = np.array(labels, dtype=np.int64)
print(f'  Total images: {len(data)}')
unique, counts = np.unique(labels, return_counts=True)
print(f'  Class distribution: min={counts.min()}, max={counts.max()}, mean={counts.mean():.0f}')


[1] Loading training data...
  Total images: 26010
  Class distribution: min=180, max=1260, mean=605


In [15]:
print('\n[2] Splitting train/val...')
indices = np.arange(len(data))
X_train, X_val, y_train, y_val = train_test_split(
    data, labels, test_size=1 - TRAIN_VAL_RATIO, random_state=SEED, stratify=labels
)
print(f'  Train: {len(X_train)}, Val: {len(X_val)}')


[2] Splitting train/val...
  Train: 20808, Val: 5202


In [16]:
print('\n[3] Computing class weights...')
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'  Weight range: {class_weights.min():.3f} - {class_weights.max():.3f}')


[3] Computing class weights...
  Weight range: 0.480 - 3.360


In [17]:
class TrafficSignDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_dataset = TrafficSignDataset(X_train, y_train)
val_dataset = TrafficSignDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [18]:
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
        )

        self._to_linear = None
        self._get_conv_output()

        self.fc = nn.Sequential(
            nn.Linear(self._to_linear, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def _get_conv_output(self):
        x = torch.zeros(1, 3, IMG_HEIGHT, IMG_WIDTH)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        self._to_linear = x.reshape(1, -1).size(1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)
        return x

model = TrafficSignCNN(num_classes=NUM_CLASSES).to(DEVICE)
print(f'\n[4] Model: {sum(p.numel() for p in model.parameters()):,} parameters')


[4] Model: 622,283 parameters


In [19]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

In [20]:
print('\n[5] Training...')
best_val_acc = 0.0
best_state = None
train_history, val_history = [], []

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_preds, train_targets = 0.0, [], []
    for images, targets in train_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_preds.append(outputs.argmax(1).cpu().numpy())
        train_targets.append(targets.cpu().numpy())

    train_loss /= len(train_dataset)
    train_preds = np.concatenate(train_preds)
    train_targets = np.concatenate(train_targets)
    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average='macro')

    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)
            val_preds.append(outputs.argmax(1).cpu().numpy())
            val_targets.append(targets.cpu().numpy())

    val_loss /= len(val_dataset)
    val_preds = np.concatenate(val_preds)
    val_targets = np.concatenate(val_targets)
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average='macro')

    scheduler.step(val_acc)
    train_history.append((train_loss, train_acc, train_f1))
    val_history.append((val_loss, val_acc, val_f1))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
        print(f'  Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}')

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(os.path.dirname(RESULT_PATH), 'best_model.pth'))
print(f'\n  Best Val Acc: {best_val_acc:.4f}')


[5] Training...
  Epoch   1/30 | Train Loss: 3.5872 Acc: 0.0672 F1: 0.0500 | Val Loss: 3.1251 Acc: 0.1190 F1: 0.0802
  Epoch   5/30 | Train Loss: 0.9059 Acc: 0.6915 F1: 0.6919 | Val Loss: 0.3176 Acc: 0.8787 F1: 0.9123
  Epoch  10/30 | Train Loss: 0.1178 Acc: 0.9659 F1: 0.9683 | Val Loss: 0.0155 Acc: 0.9925 F1: 0.9955
  Epoch  15/30 | Train Loss: 0.0668 Acc: 0.9809 F1: 0.9799 | Val Loss: 0.0074 Acc: 0.9969 F1: 0.9982
  Epoch  20/30 | Train Loss: 0.0431 Acc: 0.9861 F1: 0.9857 | Val Loss: 0.0044 Acc: 0.9985 F1: 0.9992
  Epoch  25/30 | Train Loss: 0.0341 Acc: 0.9904 F1: 0.9897 | Val Loss: 0.0077 Acc: 0.9967 F1: 0.9973
  Epoch  30/30 | Train Loss: 0.0158 Acc: 0.9959 F1: 0.9960 | Val Loss: 0.0042 Acc: 0.9983 F1: 0.9980

  Best Val Acc: 0.9987


In [21]:
print('\n[6] Test inference...')
test_dir = os.path.join(DATA_ROOT, 'Test')
result_df = pd.read_csv(RESULT_PATH)
test_files = sorted(result_df['id'].values)

test_batches, test_names = [], []
for i, fname in enumerate(test_files):
    img_path = os.path.join(test_dir, str(fname))
    img = cv2.imread(img_path)
    if img is None:
        print(f'  Warning: cannot read {fname}')
        test_batches.append(np.zeros((IMG_HEIGHT, IMG_WIDTH, CHANNELS), dtype=np.float32))
    else:
        img = Image.fromarray(img, 'RGB')
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))
        test_batches.append(np.array(img, dtype=np.float32) / 255.0)

test_batches = np.array(test_batches)
test_tensor = torch.tensor(test_batches, dtype=torch.float32).permute(0, 3, 1, 2)

model.eval()
predictions = []
with torch.no_grad():
    for i in range(0, len(test_tensor), BATCH_SIZE):
        batch = test_tensor[i:i + BATCH_SIZE].to(DEVICE)
        outputs = model(batch)
        predictions.append(outputs.argmax(1).cpu().numpy())
predictions = np.concatenate(predictions)

result_df['class'] = predictions
result_df.to_csv(RESULT_PATH, index=False)
print(f'  Saved {len(predictions)} predictions to {RESULT_PATH}')

print('\n[DONE]')
print(f'  Train  Acc: {train_history[-1][1]:.4f}, F1: {train_history[-1][2]:.4f}')
print(f'  Val    Acc: {val_history[-1][1]:.4f}, F1: {val_history[-1][2]:.4f}')
print(f'  Best   Acc: {best_val_acc:.4f}')


[6] Test inference...
  Saved 8670 predictions to C:\hong\python-ws\project-1\result.csv

[DONE]
  Train  Acc: 0.9959, F1: 0.9960
  Val    Acc: 0.9983, F1: 0.9980
  Best   Acc: 0.9987
